# **Variational Autoencoder**

![alt text](notebook-images/variational-autoencoder.png)

---

## Why do we even need **variational autoencoder** to generate new data?

* **First approach**, would be to randomly sample the data from latent space and run them through the decoder. But here's the catch. This approach won't produce useful results, mainly because the latent space is disorganised and irregular.

* **Another approach** would be to take an encoded image and then sample points that are in the neighborhood of it's latent representation. This should give us images, that resemble the original, right? Not exactly. Latent space is so poorly structured, that even nearby points often don't correspond to meaningful variations of original image. If we sample close to the original image, we end up with the same reconstruction.

> **Ideally, we want to organize the latent space**

---

## Difference between **variational autoencoder** and classic autoencoder

In a standard autoencoder, representation is a single point in the latent space. Now with **variational autoencoders**, things work a bit differently. Instead of mapping the input to the single point, the encoder converts the input into a probability distribution, that we choose to be a Gaussian. Essentially, the encoder transforms the input data into the parameters of this Gaussian. Now instead representing our image as a single point in the latent space, its represented as a Gaussian distribution. From this latent distribution, we sample points at random, and the decoder converts these sampled points back into the input space. 

We can then compute the different parts of the loss and backpropagate it through the network. But to make this possible, we have to use technique called **reparameterization** to handle the randomness outside of the network. Then the process becomes differentiable with respect to the mean and variance.

![alt text](notebook-images/reparam.png)

> **The Log-Variance Trick**: In practice (e.g., in PyTorch), the encoder's final layers do not output $\sigma$ or $\sigma^2$ directly, because linear layers can output negative values, and variance must be positive. Instead, the network is trained to output the logarithm of the variance, usually denoted as $\log(\sigma^2)$. We then retrieve $\sigma$ for the reparameterization trick using the exponential function: $\sigma = \exp(0.5 \cdot \log(\sigma^2))$. This guarantees that $\sigma$ is always positive and makes training numerically stable.

---

### **Kullback-Leibler divergence**

![alt](notebook-images/kl_div.png)

> **The role of KL Divergence**: It acts as a regularizer. While the reconstruction loss forces the network to encode information well, the **KL Divergence** forces the generated distributions to be as close to the **Standard Normal Distribution** $\mathcal{N}(0, 1)$ as possible. This prevents the network from "cheating" by shrinking the variance to zero ($\sigma \to 0$). It forces the distributions of different classes to overlap smoothly around the center of our latent space, eliminating "dead zones"

We "just" have to compute this quantity, backpropagate the combined loss, update the network's weight and repeat this process for each image in the dataset.

After this process, the VAE is able to reconstruct any digit it encounters. But that's not all - the trained VAE also allows us to generate entirely new handwritten digits that weren't part of our original dataset.

**That's awesome, but this coin has two sides:**
- VAE's tend to produce blurry images
- lack of conditioning to impose specific constraints on the generated data

> **Latent Space Continuity and Interpolation**: Because the KL divergence forces the distributions to overlap smoothly, the latent space becomes continuous. This means we can pick the latent vector for digit "1", pick the vector for digit "7", and draw a straight line between them. If we sample points along this line and decode them, we will see a smooth, frame-by-frame transformation (morphing) of a 1 into a 7.

---
### Application of Variational Autoencoders
- Generating synthetic data (data augmentation)
- Smooth editing and vectorizing data (latent space arithmetic)
- Probabilistic anomaly detection - works better than classic autoencoder. VAE not only returns reconstruction error, but it also can give us probability for particular sample to occur
- Environment compression (reinforcement learning)